# പാഠം 05 - ഏജന്റിക് RAG


## സജ്ജമാക്കൽ

Microsoft Agent Framework ഉപയോഗിച്ച് Agentic RAG (Retrieval-Augmented Generation) മാതൃക ഈ നോട്ട്‌ബുക്ക് പ്രദർശിപ്പിക്കുന്നു.

**ആവശ്യമായvoorwaarden:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — നിങ്ങളുടെ Azure AI Search സേവന എ_endpoint_
- `AZURE_SEARCH_API_KEY` — നിങ്ങളുടെ Azure AI Search API കീ
- പരിസ്ഥിതി മാറ്റിട്ടിലൂടെ സ്ഥാപിച്ച Azure OpenAI വിന്യാസം
- Azure CLI യിൽ ലോഗിൻ ചെയ്തിരിക്കണം (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ഏന്റേജിക് RAG എന്നത് എന്താണ്?

പരമ്പരാഗത RAG ഒരു നിശ്ചിത പൈപ്പ്‌ലൈൻ പിന്തുടരുന്നു: രേഖകൾ തിരയുക, തുടർന്ന് ഒരു പ്രതികരണം സൃഷ്ടിക്കുക. **ഏന്റേജിക് RAG** അതിൽ കൂടുതലാണ്, ഏജന്റ് സ്വയം **എപ്പോൾ**നും **എങ്ങിനെയാണ്** വിവരം തിരയേണ്ടത് തീരുമാനിക്കാനുള്ള സ്വാതന്ത്ര്യം നൽകുന്നു.

ഏന്റേജിക് RAG ഉപയോഗിച്ചാൽ, ഏജന്റ് ഇതുകൾ ചെയ്യാൻ കഴിയും:
- ഒരു ചോദ്യത്തിന് മറുപടിയാകുന്നതിനു മുൻപ് വിവരം തിരച്ചിൽ ആവശ്യമുണ്ടോ എന്ന് **തീർച്ചുകൽ** ചെയ്യുക
- ഏതെങ്കിലും ഡേറ്റാ സ്രോതസോ ടൂളോ ചോദിക്കാനായി **തിരഞ്ഞെടുക്കുക**
- തിരഞ്ഞെടുത്ത ഫലങ്ങൾ **അവലോകനം** ചെയ്ത് പ്രഥമ ശ്രമം പോരാത്ത പക്ഷം പിന്നാലെ തിരച്ചിലുകൾ നടത്തുക
- പല തിരച്ചിൽ ഘട്ടങ്ങളിൽ നിന്നുള്ള വിവരങ്ങൾ ഏകാധിപത്യമുള്ള മറുപടിയായി **സ്‌നേഹിക്കുക**

സ്തിര retrieve-then-generate പൈപ്പ്‌ലൈൻക്കൊപ്പം താരതമ്യപ്പെടുത്തുമ്പോൾ ഇത് ഏജന്റിനെ കൂടുതൽ സാന്ദ്രവും ശരിയായവയുമാക്കുന്നു.


## Creating a Search Tool

In Agentic RAG, external data sources are wrapped as **tools** that the agent can invoke on demand. This lets the agent treat retrieval as just another action it can take, rather than a mandatory step.

Below we define a travel knowledge base and expose it as a tool the agent can call to look up destination information.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ഏജന്റ് നിർമ്മിക്കുന്നത്

ഇപ്പോൾ നാം എപ്പോഴും മറുപടി നൽകുന്നതിന് മുമ്പ് **വിവരം ശേഖരിക്കാൻ നിർദേശിച്ച** ഒരു ഏജന്റ് നിർമ്മിക്കുന്നു. ഏജന്റ് തന്റെ ഉത്തരം സ്വന്തം പരിശീലന ഡാറ്റയ്ക്ക് പ്രാതിഭാസപ്പെടാതെ വിജ്ഞാന സഞ്ചയത്തിലൂടെ നിൽക്കുന്നതിന് `search_travel_knowledge` ടൂൾ ഉപയോഗിക്കുന്നു.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ആവർത്തിച്ച് തിരയൽ — മേക്കർ-ചെക്കർ മാതൃക

Agentic RAG-ന്റെ ഒരു പ്രധാന നേട്ടമാണ് **ആവർത്തിച്ച് തിരയൽ**. ഏജന്റ് പ്രാരംഭ കണ്ടെത്തലുകൾ ഉറപ്പാക്കാനും, മെച്ചപ്പെടുത്താനും, വികസിപ്പിക്കാനുമുള്ള 여러 റൗണ്ടുകൾ തിരയലുകൾ നടത്താൻ കഴിയും — "മേക്കർ-ചെക്കർ" പ്രവൃത്തി സ്രോതസ്സിനെ പോലെ:

1. **മേക്കർ ഘട്ടം**: ഏജന്റ് പ്രാരംഭ വിവരങ്ങൾ തിരയുകയും ഒരു മറുപടി രൂപപ്പെടുത്തുകയും ചെയ്യുന്നു.
2. **ചെക്കർ ഘട്ടം**: ഏജന്റ് വിശദാംശങ്ങൾ ഉറപ്പാക്കാനോ എൻറ് ഒഴിവുകൾ നിറയ്ക്കാനോ അധിക തിരയലുകൾ നടത്തുന്നു.

താഴെ, ഏജന്റിനോട് , ക്‌ളങ്ങളും ഇടങ്ങളുമായി താരതമ്യം ചെയ്യേണ്ടതുളള ഒരു ചോദ്യം ചോദിക്കുകയും, അതിനാൽ ഏജന്റ് പലതവണ തിരയുകയും ചെയ്യുന്നു.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ Microsoft Agent Framework ഉപയോഗിച്ച് **Agentic RAG** സിസ്റ്റം നിർമ്മിക്കാൻ പഠിച്ചു:

- **Agentic RAG** ഏജന്റുകൾ സ്വാതന്ത്ര്യമായി വിവരങ്ങൾ എപ്പോൾ തിരയണമെന്ന് തീരുമാനിക്കാനാകുന്നതായി, തിരയൽ സ്ഥിരം അല്ലാതെയാണ്.
- **കൂട്ടികളായി ഉപകരണങ്ങൾ**: Azure AI Search പോലുള്ള ബാഹ്യ വിജ്ഞാന ആധാരങ്ങൾ ഉപകരണങ്ങളായി റാപ്പുചെയ്യുകയോടെ ഏജന്റ് അവ 호출 ചെയ്യാൻ കഴിയും.
- **പുനരാവൃത്തി തിരയൽ**: മേക്കർ-ചെക്കർ മാതൃക ഏജന്റിന് നിരവധി തിരയൽ റൗണ്ടുകൾ നടത്താൻ അവസരം കൊടുക്കുന്നു — തിരയൽ, പരിശോധിക്കൽ, മെച്ചപ്പെടുത്തൽ — അന്തിമ ഉത്തരം സൃഷ്ടിക്കുന്നതിന് മുമ്പായി.

ഉത്പാദന പരിസ്ഥിതിയിൽ, സ്മൃതിയിലെ `TRAVEL_KNOWLEDGE_BASE` യെ വലിയ തോതിലുള്ള യാത്രാ രേഖ തിരയലിനെ കൈകാര്യം ചെയ്യാൻ യഥാർത്ഥ Azure AI Search ഇൻഡക്സ് ഉപയോഗിച്ച് മാറ്റിവെക്കണമെന്നും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
